# QIntent SDK Quickstart

QIntent is an intent-first quantum language powered by QDSV. Users declare what they want to find, evaluate, rank, sample, or verify; QDSV decides how to represent and execute that intent.

This notebook uses the public QIntent API through the lightweight `qdsv-qintent` Python SDK. The SDK defaults to the QDSV QuEST route, which executes QIntent over state spaces without requiring users to write circuits first.

In [ ]:
!pip install -q --upgrade qdsv-qintent

In [ ]:
from qintent import QIntentClient

client = QIntentClient()
client.spec()["status"]

## Example 1: select rows by threshold

In [ ]:
rows = [
    {"candidate_index": 0, "score": 720},
    {"candidate_index": 1, "score": 910},
    {"candidate_index": 2, "score": 840},
]

result = client.run(
    'find_rows("candidate_index").where("score", ">=", 850)',
    rows=rows,
)

result["result"]["selected_rows"]

## Example 2: rank top candidates

In [ ]:
rows = [
    {"candidate_index": 0, "quality_score": 720},
    {"candidate_index": 1, "quality_score": 910},
    {"candidate_index": 2, "quality_score": 860},
    {"candidate_index": 3, "quality_score": 970},
]

source = '''
find_rows("candidate_index")
  .where("quality_score", ">=", 850)
  .rank_by("quality_score")
  .top_k(2)
'''

result = client.run(source, rows=rows)
result["result"]["selected_rows"]

## Example 3: Python-like state-space query

In [ ]:
source = '''
x = domain(0, 15)
score = clip(round(max(x, 0)), 0, 10)
find(x).where(all([score >= 7, score <= 9, x not in [8]])).rank_by(score).top_k(3)
'''

compiled = client.compile(source)
executed = client.run(source)

compiled["compiled_summary"], executed["result"].get("ranked_candidates")